# "THE POUR IS RIGHT" - Wine Pricer Capstone

## Extra: what does the ensemble cost to serve, and how fast is it?

A reader asked how the three-model ensemble affects serving cost and latency on Modal. This notebook measures it instead of guessing.

It answers four questions:

1. **Cold start**: how long the first call waits when the Modal service has gone to sleep.
2. **Warm calls**: how long a call takes once the service is awake.
3. **Latency of one ensemble estimate**: each model's share, and what calling the three models in parallel instead of one after another would save.
4. **Is the fine-tuned model worth its GPU?** The accuracy of the ensemble with and without it, on the same 200 test wines.

It finishes with the cost of each estimate and of a typical hourly scan.

## Costs and time

| Step | Cost | Time |
|---|---|---|
| Claude Sonnet 5 with RAG, 200 wines | under \$1 (the notebook measures it) | 2 to 4 minutes |
| Fine-tuned model on Modal, about 220 calls | a few cents of T4 time | 5 to 10 minutes |
| Neural network, rewrites | free / a fraction of a cent | seconds |

## Before you start

- The service must still be deployed. If the first Modal cell says it can't find `wine-pricer-service`, run `uv run modal deploy pricer_service.py` from this folder, then start again.
- **Run this notebook when nothing has called the service for at least 2 minutes**, so that the first call really is a cold start.
- Close any other notebook that has the Qdrant store open (`wines_vectorstore`).

In [ ]:
import time
import statistics
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import modal
from dotenv import load_dotenv
from huggingface_hub import HfApi
from qdrant_client import QdrantClient
from pricer.items import Item
from agents.frontier_agent import FrontierAgent
from agents.neural_network_agent import NeuralNetworkAgent
from agents.preprocessor import Preprocessor

load_dotenv(override=True)

# Claude Sonnet 5 prices, in dollars per million tokens
CLAUDE_INPUT_PER_MILLION = 2.00
CLAUDE_OUTPUT_PER_MILLION = 10.00

# Fill this in from https://modal.com/pricing (GPU tasks, Nvidia T4), in dollars per hour.
# Leave it as None to see the GPU time in seconds only.
T4_DOLLARS_PER_HOUR = None

# Modal shuts a container down 2 minutes after its last call, and bills until then
SCALEDOWN_SECONDS = 120

In [ ]:
username = HfApi().whoami()["name"]
train, val, test = Item.from_hub(f"{username}/wines_lite")
wines = test[:200]
truth = np.array([item.price for item in wines])
print(f"Loaded {len(wines)} test wines, median price ${np.median(truth):.0f}")

## Set up the models

The same three models the `EnsembleAgent` uses. We count Claude's tokens as we go: the cell wraps Claude's `messages.create` so that every call adds its token usage to a running total.

In [ ]:
client = QdrantClient(path="wines_vectorstore")
frontier = FrontierAgent(client)
neural_network = NeuralNetworkAgent()
preprocessor = Preprocessor()

claude_usage = {"calls": 0, "input": 0, "output": 0}
original_create = frontier.claude.messages.create

def counting_create(*args, **kwargs):
    response = original_create(*args, **kwargs)
    claude_usage["calls"] += 1
    claude_usage["input"] += response.usage.input_tokens
    claude_usage["output"] += response.usage.output_tokens
    return response

frontier.claude.messages.create = counting_create

Pricer = modal.Cls.from_name("wine-pricer-service", "Pricer")
pricer = Pricer()
print("Ready")

## 1. Cold start

Nothing has called the service for a while, so Modal has to start a container with a T4, load the 4-bit base model from the volume, and attach our adapter before it can answer. You can watch it at https://modal.com/apps.

In [ ]:
start = time.perf_counter()
answer = pricer.price.remote(wines[0].summary)
cold_seconds = time.perf_counter() - start
print(f"Cold call: {cold_seconds:.1f} seconds (answer ${answer:.2f}, real price ${wines[0].price:.2f})")

## 2. Warm calls

The container is awake now. Ten more calls show the normal speed. We report the median, which ignores the odd slow network call.

In [ ]:
warm_times = []
for item in wines[1:11]:
    start = time.perf_counter()
    pricer.price.remote(item.summary)
    warm_times.append(time.perf_counter() - start)

warm_seconds = statistics.median(warm_times)
print(f"Warm call: median {warm_seconds:.2f} s, fastest {min(warm_times):.2f} s, slowest {max(warm_times):.2f} s")

## 3. Where the time goes in one estimate

In production, an estimate has four steps:

1. **Rewrite**: the deal's text is rewritten into our six-line summary format (gpt-oss-20b on Groq).
2. **Frontier**: the 5 most similar wines are looked up in Qdrant, then Claude is asked with them as context.
3. **Specialist**: the fine-tuned model on Modal.
4. **Neural network**: on this laptop's CPU.

The `EnsembleAgent` runs steps 2 to 4 one after another, so their times add up. If it ran them at the same time, the wait would be only as long as the slowest. We time each step on 10 wines, then time the parallel version for real.

In [ ]:
def timed(function, *args):
    start = time.perf_counter()
    result = function(*args)
    return result, time.perf_counter() - start

steps = {"rewrite (Groq)": [], "RAG search (Qdrant)": [], "Claude": [], "specialist (Modal, warm)": [], "neural network (CPU)": []}

for item in wines[11:21]:
    _, t = timed(preprocessor.preprocess, item.summary); steps["rewrite (Groq)"].append(t)
    (documents, prices), t = timed(frontier.find_similars, item.summary); steps["RAG search (Qdrant)"].append(t)
    _, t = timed(frontier.ask, frontier.messages_for(item.summary, documents, prices)); steps["Claude"].append(t)
    _, t = timed(pricer.price.remote, item.summary); steps["specialist (Modal, warm)"].append(t)
    _, t = timed(neural_network.price, item.summary); steps["neural network (CPU)"].append(t)

medians = {name: statistics.median(times) for name, times in steps.items()}
for name, seconds in medians.items():
    print(f"{name:26s} {seconds:6.2f} s")

frontier_seconds = medians["RAG search (Qdrant)"] + medians["Claude"]
models = [frontier_seconds, medians["specialist (Modal, warm)"], medians["neural network (CPU)"]]
print()
print(f"One after another (as now):  rewrite + {sum(models):.2f} s = {medians['rewrite (Groq)'] + sum(models):.2f} s")
print(f"In parallel (expected):      rewrite + {max(models):.2f} s = {medians['rewrite (Groq)'] + max(models):.2f} s")
print(f"Without the specialist:      rewrite + {max(frontier_seconds, medians['neural network (CPU)']):.2f} s (in parallel)")

Now the parallel version for real, on the same 10 wines: three threads, one per model, so the wait is the slowest of the three.

In [ ]:
def ensemble_parallel(summary):
    with ThreadPoolExecutor(max_workers=3) as pool:
        f = pool.submit(frontier.price, summary)
        s = pool.submit(pricer.price.remote, summary)
        n = pool.submit(neural_network.price, summary)
        return 0.8 * f.result() + 0.1 * s.result() + 0.1 * n.result()

parallel_times = [timed(ensemble_parallel, item.summary)[1] for item in wines[11:21]]
print(f"Ensemble in parallel, measured: median {statistics.median(parallel_times):.2f} s (plus the rewrite)")

## 4. Accuracy with and without the fine-tuned model

The fine-tuned model has only 10% of the ensemble's weight, but it brings all of the GPU cost and the cold start. Does it earn its place?

We price all 200 test wines with each model once, then combine the same predictions in different ways, so every row below is compared on equal terms.

- Claude is called 4 at a time to save time. The Qdrant lookups are done first, one by one, because the local Qdrant store prefers a single user.
- The specialist is called one at a time, so a single warm container does all the work. We time it to get the GPU seconds used.

In [ ]:
similars = [frontier.find_similars(item.summary) for item in wines]

def claude_price(i):
    documents, prices = similars[i]
    return frontier.get_price(frontier.ask(frontier.messages_for(wines[i].summary, documents, prices)))

with ThreadPoolExecutor(max_workers=4) as pool:
    f = np.array(list(pool.map(claude_price, range(len(wines)))))
print(f"Claude done: {claude_usage['calls']} calls so far")

start = time.perf_counter()
s = np.array([pricer.price.remote(item.summary) for item in wines])
specialist_busy_seconds = time.perf_counter() - start
print(f"Specialist done: {specialist_busy_seconds:.0f} s for {len(wines)} wines")

n = np.array([neural_network.price(item.summary) for item in wines])
print("Neural network done")

In [ ]:
def error(prediction):
    return np.abs(prediction - truth).mean()

BASELINE = 21.71   # always predict the training average

variants = {
    "Claude with RAG alone": f,
    "Fine-tuned model alone": s,
    "Neural network alone": n,
    "Ensemble 0.8 / 0.1 / 0.1 (as deployed)": 0.8 * f + 0.1 * s + 0.1 * n,
    "Without fine-tuned: Claude 0.89 / NN 0.11": (0.8 * f + 0.1 * n) / 0.9,
    "Without neural network: Claude 0.89 / FT 0.11": (0.8 * f + 0.1 * s) / 0.9,
}

print(f"{'':48s} {'error':>8s} {'vs baseline':>12s}")
for name, prediction in variants.items():
    e = error(prediction)
    print(f"{name:48s} ${e:7.2f} {100 * (BASELINE - e) / BASELINE:10.0f}% lower")

Claude's answers vary a little from run to run, so these numbers can differ by a few cents from the \$7.96 in week 8 day 2. What matters is the gap between the rows, which all use the same predictions.

As a rough guide, with 200 wines a difference smaller than about 20 to 30 cents is within the noise.

## 5. What it costs

**Claude** is billed per token. **Modal** bills a container from the moment it starts until it shuts down, and that includes the 2 idle minutes before it scales to zero. So for a short burst of calls, the idle tail is most of the bill.

The hourly scan prices up to 5 deals. With scale-to-zero, each scan pays for one cold start, 5 warm calls and the 2-minute idle tail.

In [ ]:
claude_per_estimate = (
    claude_usage["input"] * CLAUDE_INPUT_PER_MILLION + claude_usage["output"] * CLAUDE_OUTPUT_PER_MILLION
) / 1_000_000 / claude_usage["calls"]
print(f"Claude: {claude_usage['calls']} calls, average {claude_usage['input'] / claude_usage['calls']:.0f} tokens in "
      f"and {claude_usage['output'] / claude_usage['calls']:.0f} out, ${claude_per_estimate:.4f} per estimate")

DEALS_PER_SCAN = 5
gpu_per_scan = cold_seconds + DEALS_PER_SCAN * warm_seconds + SCALEDOWN_SECONDS
gpu_busy_share = DEALS_PER_SCAN * warm_seconds / gpu_per_scan
print(f"\nGPU time per hourly scan: {gpu_per_scan:.0f} s "
      f"(cold start {cold_seconds:.0f} + {DEALS_PER_SCAN} calls {DEALS_PER_SCAN * warm_seconds:.0f} + idle {SCALEDOWN_SECONDS}), "
      f"busy {100 * gpu_busy_share:.0f}% of it")
print(f"GPU time per day at 24 scans: {24 * gpu_per_scan / 60:.0f} minutes, versus 1,440 minutes for an always-on container")
print(f"Claude per scan: ${DEALS_PER_SCAN * claude_per_estimate:.3f}, per day: ${24 * DEALS_PER_SCAN * claude_per_estimate:.2f}")

if T4_DOLLARS_PER_HOUR:
    per_second = T4_DOLLARS_PER_HOUR / 3600
    print(f"\nModal per scan: ${gpu_per_scan * per_second:.3f}, per day: ${24 * gpu_per_scan * per_second:.2f}, "
          f"always on: ${24 * T4_DOLLARS_PER_HOUR:.2f} per day")
    print(f"This notebook's specialist calls: about ${(cold_seconds + specialist_busy_seconds + SCALEDOWN_SECONDS) * per_second:.2f}")
else:
    print("\nSet T4_DOLLARS_PER_HOUR in the first cell to turn GPU seconds into dollars, or check the app's usage at https://modal.com/apps")

## Wrap up

The container goes back to sleep by itself 2 minutes after the last call. Close the vector store so other notebooks can open it.

Then copy the printed results from sections 1 to 5 into the chat, and we will write the reply from real numbers.

In [ ]:
client.close()